# 🚀 Simulación de Gemelo Digital e IIoT para Monitoreo Predictivo de Maquinaria Industrial
**Asignatura:** Teoría de Sistemas (TSI115) - Universidad de El Salvador  
**Proyecto:** Tarea de Investigación - Etapa 2  
**Tema:** Diseño de un Gemelo Digital basado en IoT para el monitoreo predictivo del rendimiento en maquinaria industrial  
**Dataset de Referencia:** NASA IMS Bearing Dataset (35 Días / 840 Horas de ensayo acelerado hasta la falla)  
---
### 📌 Descripción del Cuaderno
Este cuaderno de Jupyter / Google Colab implementa la simulación computacional del **Gemelo Digital** calibrada con los mismos datos experimentales, escalas y métricas del **informe de investigación (Etapa 2)**:
1. **Entidad Observable (OME):** Telemetría IoT en tiempo real (vibración triaxial en $g$, velocidad equivalente en $mm/s$ y temperatura en $^\circ	ext{C}$) durante 840 horas (35 días).
2. **Entidad de Gemelo Digital (DTE - Caja Blanca):** Procesamiento de señales en el dominio de la frecuencia mediante la **Transformada Rápida de Fourier (FFT)** para detectar el fallo en la pista exterior **BPFO = 236.4 Hz** (2000 RPM / 33.33 Hz).
3. **Evaluación de Severidad (ISO 10816):** Clasificación del estado operativo por zonas de alarma ($A$: Excelente, $B$: Permisible, $C$: Alarma, $D$: Peligro).
4. **Cibernética y Retroalimentación (Feedback Loop):** Estimación de Vida Útil Remanente (RUL), Índice de Salud ($HI\%$) y disparo automático de alertas preventivas para preservar la **homeostasis** del activo.


In [ ]:
# ========================================== #
# 1. IMPORTACIÓN DE LIBRERÍAS Y CONFIGURACIÓN #
# ========================================== #
import numpy as np
import pandas as pd
import scipy.fft as fft
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Estilo visual para los gráficos
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10
print('✅ Librerías cargadas exitosamente.')


## 📡 1. Emulador de Telemetría IoT (Entidad OME)
Simulación de la telemetría del ensayo **NASA IMS Bearing Dataset** durante **840 horas (35 días)** de operación continua.
* **Frecuencia de muestreo ($F_s$):** 20,000 Hz
* **Velocidad de rotación ($1X$):** 2000 RPM ($33.33$ Hz)
* **Frecuencia de falla en pista exterior (BPFO):** $236.4$ Hz (Rexnord ZA-2115)


In [ ]:
# ========================================== #
# 2. GENERACIÓN DE TELEMETRÍA NASA IMS (840h) #
# ========================================== #
np.random.seed(42)

# Parámetros físicos exactos del informe (NASA IMS)
fs = 20000            # Frecuencia de muestreo (20 kHz)
duracion_muestra = 0.5# Segundos por ventana
num_muestras_t = int(fs * duracion_muestra)
t = np.linspace(0, duracion_muestra, num_muestras_t, endpoint=False)

# Frecuencias mecánicas características (2000 RPM)
f_rot = 2000 / 60.0    # 1X = 33.33 Hz
f_bpfo = 236.4        # BPFO = 236.4 Hz

# 840 horas de simulación (35 días de ensayo real NASA IMS)
horas = np.arange(1, 841)
historial_rms_g = []
historial_v_mms = []
historial_temp = []
historial_kurtosis = []
muestras_tiempo_seleccionadas = {}

for h in horas:
    # Tasa de degradación calibrada con las 4 fases del informe de investigación
    if h <= 500:
        # Fase 1: Operación Normal (0.025g - 0.035g)
        degradacion = 0.015 + 0.010 * (h / 500)
    elif h <= 680:
        # Fase 2: Degradación Incipiente (0.035g - 0.085g)
        degradacion = 0.025 + 0.085 * ((h - 500) / 180)**1.3
    elif h <= 780:
        # Fase 3: Alerta Severa / Retroalimentación (0.085g - 0.150g)
        degradacion = 0.110 + 0.085 * ((h - 680) / 100)**1.5
    else:
        # Fase 4: Falla Inminente / Parada de Emergencia (> 0.150g)
        degradacion = 0.200 + 0.350 * ((h - 780) / 60)**2.0
        
    # Componentes físicas de la señal:
    # 1. Rotación 1X fundamental
    s_normal = 0.025 * np.sin(2 * np.pi * f_rot * t)
    
    # 2. Impacto de falla BPFO modulado por velocidad
    s_falla = degradacion * np.sin(2 * np.pi * f_bpfo * t) * (1 + 0.35 * np.sin(2 * np.pi * f_rot * t))
    
    # 3. Ruido blanco gaussiano industrial
    ruido = np.random.normal(0, 0.008, num_muestras_t)
    
    # Señal total en aceleración (g)
    senos_acc = s_normal + s_falla + ruido
    
    # Cálculo del valor RMS en g
    acc_rms = np.sqrt(np.mean(senos_acc**2))
    
    # Conversión a velocidad equivalente RMS mm/s
    v_mms = acc_rms * 9.81 * 1000 / (2 * np.pi * f_rot) * 0.14
    
    # Evolución térmica correlacionada (°C)
    temp = 42.0 + 0.015 * h + 45.0 * (degradacion**1.2) + np.random.normal(0, 0.2)
    
    # Kurtosis
    kurt = np.mean((senos_acc - np.mean(senos_acc))**4) / (np.std(senos_acc)**4)
    
    historial_rms_g.append(acc_rms)
    historial_v_mms.append(v_mms)
    historial_temp.append(temp)
    historial_kurtosis.append(kurt)
    
    # Guardar ventanas para análisis FFT en horas clave (100h, 600h, 720h, 820h)
    if h in [100, 600, 720, 820]:
        muestras_tiempo_seleccionadas[h] = (t, senos_acc)

historial_rms_g = np.array(historial_rms_g)
historial_v_mms = np.array(historial_v_mms)
historial_temp = np.array(historial_temp)
historial_kurtosis = np.array(historial_kurtosis)

print(f'✅ Telemetría emulada correctamente para {len(horas)} horas de ensayo (35 días).')


## 🔬 2. Procesamiento de Señales en el Gemelo Digital (DTE - Caja Blanca)
Transformamos la señal de vibración del dominio del tiempo al espectro de frecuencias mediante la **Transformada Rápida de Fourier (FFT)**. Esto permite pasar de un enfoque de **caja negra** a una **caja blanca**, identificando la firma espectral exacta de la falla **BPFO = 236.4 Hz**.


In [ ]:
# ========================================== #
# 3. ANALÍTICA DE SEÑAL Y CÁLCULO DE FFT     #
# ========================================== #
def calcular_fft(senos_acc, fs):
    N = len(senos_acc)
    yf = fft.fft(senos_acc)
    xf = fft.fftfreq(N, 1 / fs)[:N//2]
    amplitud = 2.0 / N * np.abs(yf[0:N//2])
    return xf, amplitud

# Gráfico de las 4 fases de degradación (FFT)
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
horas_clave = [100, 600, 720, 820]
titulos = [
    'Fase 1: Hora 100 - Estado Normal (1X = 33.3 Hz)',
    'Fase 2: Hora 600 - Degradación Incipiente (BPFO = 236.4 Hz)',
    'Fase 3: Hora 720 - Alerta Severa (BPFO + Armónicos)',
    'Fase 4: Hora 820 - Peligro / Falla Inminente (Ruido de Banda Ancha)'
]

for idx, h in enumerate(horas_clave):
    ax = axes[idx//2, idx%2]
    t_m, s_m = muestras_tiempo_seleccionadas[h]
    xf, amplitud = calcular_fft(s_m, fs)
    
    # Limitar visualización a 0 - 800 Hz
    mask = xf <= 800
    ax.plot(xf[mask], amplitud[mask], color='#1f77b4' if idx < 2 else '#d62728', lw=1.2)
    ax.axvline(x=f_bpfo, color='orange', linestyle='--', alpha=0.8, label=f'BPFO ({f_bpfo} Hz)')
    ax.axvline(x=f_rot, color='green', linestyle=':', alpha=0.8, label=f'1X ({f_rot:.1f} Hz)')
    ax.set_title(titulos[idx], fontweight='bold')
    ax.set_xlabel('Frecuencia (Hz)')
    ax.set_ylabel('Amplitud (g)')
    ax.legend(loc='upper right')
    ax.set_ylim(0, max(0.08, np.max(amplitud[mask]) * 1.15))

plt.tight_layout()
plt.show()


## ⚙️ 3. Evaluación de Severidad ISO 10816 y Alertas Cibernéticas
Clasificamos el estado de operación según los umbrales de la norma **ISO 10816** para maquinaria industrial en aceleración ($g$):
* 🟢 **Zona A (Bueno):** $RMS < 0.035$ g ($< 1.8$ mm/s) — Operación normal.
* 🟡 **Zona B (Permisible):** $0.035 \le RMS < 0.085$ g ($1.8 - 4.5$ mm/s) — Operación continua sin restricciones.
* 🟠 **Zona C (Alarma):** $0.085 \le RMS < 0.150$ g ($4.5 - 7.1$ mm/s) — Disparo de retroalimentación cibernética.
* 🔴 **Zona D (Peligro):** $RMS \ge 0.150$ g ($> 7.1$ mm/s) — Riesgo de daño catastrófico / Parada de emergencia.


In [ ]:
# ========================================== #
# 4. CIBERNÉTICA, RETROALIMENTACIÓN Y RUL    #
# ========================================== #
# Clasificación ISO 10816
zonas_iso = []
indice_salud = []
rul_horas = []

# Umbrales exactos del informe (en g)
umb_b = 0.035
umb_c = 0.085
umb_d = 0.150

hora_alerta_ciber = None
hora_falla_critica = None

for idx, h in enumerate(horas):
    rms = historial_rms_g[idx]
    
    if rms < umb_b:
        zona = 'Zona A (Bueno)'
        hi = 100.0 - (rms / umb_b) * 15.0
    elif rms < umb_c:
        zona = 'Zona B (Permisible)'
        hi = 85.0 - ((rms - umb_b) / (umb_c - umb_b)) * 25.0
    elif rms < umb_d:
        zona = 'Zona C (Alarma)'
        hi = 60.0 - ((rms - umb_c) / (umb_d - umb_c)) * 40.0
        if hora_alerta_ciber is None:
            hora_alerta_ciber = h
    else:
        zona = 'Zona D (Peligro)'
        hi = max(0.0, 20.0 - ((rms - umb_d) / 0.250) * 20.0)
        if hora_falla_critica is None:
            hora_falla_critica = h
            
    # Estimación de RUL (Horas restantes)
    rul = max(0, 840 - h)
    
    zonas_iso.append(zona)
    indice_salud.append(hi)
    rul_horas.append(rul)

print(f'🚨 Disparo de Alerta Cibernética (Zona C - Retroalimentación): Hora {hora_alerta_ciber} (Día {hora_alerta_ciber//24})')
print(f'🛑 Estado Crítico de Falla (Zona D - Parada Recomendada): Hora {hora_falla_critica} (Día {hora_falla_critica//24})')


## 📊 4. Dashboard del Gemelo Digital (Entidad UE)
Construimos el panel de visualización integral de 4 gráficos sincronizados exactos al informe de investigación.


In [ ]:
# ========================================== #
# 5. DASHBOARD INTEGRAL DE MONITOREO        #
# ========================================== #
fig, axs = plt.subplots(2, 2, figsize=(15, 10))

# Graph 1: Tendencia Vibratoria RMS (g) e ISO 10816
axs[0, 0].plot(horas, historial_rms_g, color='#1f77b4', lw=2, label='Vibración RMS (g)')
axs[0, 0].axhline(y=umb_b, color='green', linestyle='--', alpha=0.7, label='Límite Zona B (0.035 g)')
axs[0, 0].axhline(y=umb_c, color='orange', linestyle='--', alpha=0.7, label='Límite Zona C (0.085 g)')
axs[0, 0].axhline(y=umb_d, color='red', linestyle='--', alpha=0.7, label='Límite Zona D (0.150 g)')
if hora_alerta_ciber:
    axs[0, 0].axvline(x=hora_alerta_ciber, color='orange', linestyle=':', lw=2, label=f'Alerta Ciber (h={hora_alerta_ciber})')
axs[0, 0].set_title('1. Tendencia de Vibración RMS (g) - ISO 10816', fontweight='bold')
axs[0, 0].set_xlabel('Tiempo de Operación (Horas)')
axs[0, 0].set_ylabel('Aceleración RMS (g)')
axs[0, 0].legend(loc='upper left', fontsize=8)

# Graph 2: Evolución Térmica (°C)
axs[0, 1].plot(horas, historial_temp, color='#ff7f0e', lw=2, label='Temperatura Carcasa (°C)')
axs[0, 1].axhline(y=60, color='orange', linestyle='--', alpha=0.7, label='Temperatura Advertencia (60 °C)')
axs[0, 1].axhline(y=80, color='red', linestyle='--', alpha=0.7, label='Temperatura Crítica (80 °C)')
axs[0, 1].set_title('2. Evolución Térmica del Activo (°C)', fontweight='bold')
axs[0, 1].set_xlabel('Tiempo de Operación (Horas)')
axs[0, 1].set_ylabel('Temperatura (°C)')
axs[0, 1].legend(loc='upper left', fontsize=8)

# Graph 3: Índice de Salud del Activo (Health Index - HI %)
axs[1, 0].plot(horas, indice_salud, color='#2ca02c', lw=2, label='Índice de Salud HI (%)')
axs[1, 0].axhline(y=60, color='orange', linestyle='--', alpha=0.7, label='Umbral Mantenimiento (60%)')
axs[1, 0].axhline(y=20, color='red', linestyle='--', alpha=0.7, label='Umbral Crítico (20%)')
axs[1, 0].fill_between(horas, indice_salud, color='#2ca02c', alpha=0.15)
axs[1, 0].set_title('3. Índice de Salud Computado (HI %)', fontweight='bold')
axs[1, 0].set_xlabel('Tiempo de Operación (Horas)')
axs[1, 0].set_ylabel('Salud (%)')
axs[1, 0].legend(loc='lower left', fontsize=8)

# Graph 4: Estimación de Vida Útil Remanente (RUL Horas)
axs[1, 1].plot(horas, rul_horas, color='#9467bd', lw=2, label='Proyección RUL (Horas)')
axs[1, 1].set_title('4. Estimación de Vida Útil Remanente (RUL)', fontweight='bold')
axs[1, 1].set_xlabel('Tiempo de Operación (Horas)')
axs[1, 1].set_ylabel('Horas Remanentes')
axs[1, 1].legend(loc='upper right', fontsize=8)

plt.tight_layout()
plt.show()


## 📋 5. Tabla Resumen de Fases y Resultados Coincidentes
Presentamos la tabla resumida de las fases operativas de la simulación para contrastación directa con el informe de investigación.


In [ ]:
# ========================================== #
# 6. TABLA RESUMEN DE RESULTADOS            #
# ========================================== #
df_resumen = pd.DataFrame({
    'Hora': horas,
    'Vibración RMS (g)': np.round(historial_rms_g, 3),
    'Velocidad RMS (mm/s)': np.round(historial_v_mms, 2),
    'Temp (°C)': np.round(historial_temp, 1),
    'Estado ISO 10816': zonas_iso,
    'Salud HI (%)': np.round(indice_salud, 1),
    'RUL (Horas)': rul_horas
})

# Horas representativas coincidiendo exactamente con el informe de investigación
horas_muestra = [100, 550, 680, 780, 840]
tabla_final = df_resumen[df_resumen['Hora'].isin(horas_muestra)].reset_index(drop=True)
print('✅ Dashboard generado y sincronizado correctamente con el Informe:')
print(tabla_final.to_string(index=False))
